# Course Recommendation System
> Recommend Similiar Courses for which the user is searching/looking for

Steps involved
1. Import the data from the dataset
2. Perform Text Processing and EDA and generate insights
3. After Converting Text to numeric values, now calculate the cosine similiarity score
4. After finding the similiarity score, sort the valus which have similiar similiarity score and recommend the course
5. Integrate the Recommendation System with Flask Framework
6. Deploy the Application on Heroku Cloud Platform

### Tools & Technologies: NLP, ML, JS, Flask

## 1.

NeatText:a simple NLP package for cleaning textual data and text preprocessing. Simplifying Text Cleaning For NLP & ML

In [2]:
# importing libraries
import pandas as pd
import numpy as np
import neattext.functions as nfx
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv("udemy_course_data.csv")

In [4]:
df.head()

,course_id,course_title,url,is_paid,price,num_subscribers,num_reviews,num_lectures,level,content_duration,published_timestamp,subject,profit,published_date,published_time,year,month,day
0,1070968,Ultimate Investment Banking Course,https://www.udemy.com/ultimate-investment-bank...,True,200,2147,23,51,All Levels,1.5 hours,2017-01-18T20:58:58Z,Business Finance,429400,2017-01-18,20:58:58Z,2017,1,18
1,1113822,Complete GST Course & Certification - Grow You...,https://www.udemy.com/goods-and-services-tax/,True,75,2792,923,274,All Levels,39 hours,2017-03-09T16:34:20Z,Business Finance,209400,2017-03-09,16:34:20Z,2017,3,9
2,1006314,Financial Modeling for Business Analysts and C...,https://www.udemy.com/financial-modeling-for-b...,True,45,2174,74,51,Intermediate Level,2.5 hours,2016-12-19T19:26:30Z,Business Finance,97830,2016-12-19,19:26:30Z,2016,12,19
3,1210588,Beginner to Pro - Financial Analysis in Excel ...,https://www.udemy.com/complete-excel-finance-c...,True,95,2451,11,36,All Levels,3 hours,2017-05-30T20:07:24Z,Business Finance,232845,2017-05-30,20:07:24Z,2017,5,30
4,1011058,How To Maximize Your Profits Trading Options,https://www.udemy.com/how-to-maximize-your-pro...,True,200,1276,45,26,Intermediate Level,2 hours,2016-12-13T14:57:18Z,Business Finance,255200,2016-12-13,14:57:18Z,2016,12,13


## 2.

Whenever we will have text, we will remove stopwords, special_characters

In [5]:
df['Clean_title'] = df['course_title'].apply(nfx.remove_stopwords)

df['Clean_title'] = df['Clean_title'].apply(nfx.remove_special_characters)

df.head()

,course_id,course_title,url,is_paid,price,num_subscribers,num_reviews,num_lectures,level,content_duration,published_timestamp,subject,profit,published_date,published_time,year,month,day,Clean_title
0,1070968,Ultimate Investment Banking Course,https://www.udemy.com/ultimate-investment-bank...,True,200,2147,23,51,All Levels,1.5 hours,2017-01-18T20:58:58Z,Business Finance,429400,2017-01-18,20:58:58Z,2017,1,18,Ultimate Investment Banking Course
1,1113822,Complete GST Course & Certification - Grow You...,https://www.udemy.com/goods-and-services-tax/,True,75,2792,923,274,All Levels,39 hours,2017-03-09T16:34:20Z,Business Finance,209400,2017-03-09,16:34:20Z,2017,3,9,Complete GST Course Certification Grow Practice
2,1006314,Financial Modeling for Business Analysts and C...,https://www.udemy.com/financial-modeling-for-b...,True,45,2174,74,51,Intermediate Level,2.5 hours,2016-12-19T19:26:30Z,Business Finance,97830,2016-12-19,19:26:30Z,2016,12,19,Financial Modeling Business Analysts Consultants
3,1210588,Beginner to Pro - Financial Analysis in Excel ...,https://www.udemy.com/complete-excel-finance-c...,True,95,2451,11,36,All Levels,3 hours,2017-05-30T20:07:24Z,Business Finance,232845,2017-05-30,20:07:24Z,2017,5,30,Beginner Pro Financial Analysis Excel 2017
4,1011058,How To Maximize Your Profits Trading Options,https://www.udemy.com/how-to-maximize-your-pro...,True,200,1276,45,26,Intermediate Level,2 hours,2016-12-13T14:57:18Z,Business Finance,255200,2016-12-13,14:57:18Z,2016,12,13,Maximize Profits Trading Options


In [6]:
df.describe()

,course_id,price,num_subscribers,num_reviews,num_lectures,profit,year,month,day
count,3.683000e+03,3683.000000,3683.000000,3683.000000,3683.000000,3.683000e+03,3683.000000,3683.000000,3683.000000
mean,6.764546e+05,65.992398,3193.371165,156.448004,40.062178,2.402885e+05,2015.433342,6.162639,15.841162
std,3.437217e+05,60.985586,9498.231406,935.078241,50.366788,1.000760e+06,1.185920,3.379314,8.780906
min,8.324000e+03,0.000000,0.000000,0.000000,0.000000,0.000000e+00,2011.000000,1.000000,1.000000
25%,4.077270e+05,20.000000,110.000000,4.000000,15.000000,1.567500e+03,2015.000000,3.000000,8.000000
50%,6.882440e+05,45.000000,911.000000,18.000000,25.000000,2.305000e+04,2016.000000,6.000000,16.000000
75%,9.617290e+05,95.000000,2537.500000,67.000000,45.000000,1.182600e+05,2016.000000,9.000000,23.000000
max,1.282064e+06,200.000000,268923.000000,27445.000000,779.000000,2.431680e+07,2017.000000,12.000000,31.000000


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   course_id            3683 non-null   int64
 1   course_title         3683 non-null   str  
 2   url                  3683 non-null   str  
 3   is_paid              3683 non-null   bool 
 4   price                3683 non-null   int64
 5   num_subscribers      3683 non-null   int64
 6   num_reviews          3683 non-null   int64
 7   num_lectures         3683 non-null   int64
 8   level                3683 non-null   str  
 9   content_duration     3683 non-null   str  
 10  published_timestamp  3683 non-null   str  
 11  subject              3683 non-null   str  
 12  profit               3683 non-null   int64
 13  published_date       3683 non-null   str  
 14  published_time       3682 non-null   str  
 15  year                 3683 non-null   int64
 16  month                3683 non-null 

### Vectorize the Clean Title

In [8]:
countvect = CountVectorizer()
cvmat = countvect.fit_transform(df['Clean_title'])
cvmat

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 18364 stored elements and shape (3683, 3564)>

In [9]:
cvmat.todense()

matrix([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(3683, 3564))

## 3.

### Cosine Similiarity

In [10]:
cos_sim = cosine_similarity(cvmat)
cos_sim

array([[1.        , 0.20412415, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.20412415, 1.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.23570226],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.23570226, 0.        ,
        1.        ]], shape=(3683, 3683))

In [11]:
cos_sim.shape

(3683, 3683)

## 4. 
> After finding the similiarity score, sort the valus which have similiar similiarity score and recommend the course

### Recommend Course

In [12]:
df['course_title']

0                      Ultimate Investment Banking Course
1       Complete GST Course & Certification - Grow You...
2       Financial Modeling for Business Analysts and C...
3       Beginner to Pro - Financial Analysis in Excel ...
4            How To Maximize Your Profits Trading Options
                              ...                        
3678    Learn jQuery from Scratch - Master of JavaScri...
3679    How To Design A WordPress Website With No Codi...
3680                        Learn and Build using Polymer
3681    CSS Animations: Create Amazing Effects on Your...
3682    Using MODX CMS to Build Websites: A Beginner's...
Name: course_title, Length: 3683, dtype: str

In [13]:
title = 'How To Maximize Your Profits Trading Options'

In [14]:
course_index = pd.Series(df.index, index=df['course_title']).drop_duplicates()

course_index[title]

np.int64(4)

In [15]:
index = course_index['How To Maximize Your Profits Trading Options']
index

np.int64(4)

In [16]:
scores = list(enumerate(cos_sim[index]))
scores

[(0, np.float64(0.0)),
 (1, np.float64(0.0)),
 (2, np.float64(0.0)),
 (3, np.float64(0.0)),
 (4, np.float64(1.0)),
 (5, np.float64(0.20412414523193154)),
 (6, np.float64(0.20412414523193154)),
 (7, np.float64(0.1889822365046136)),
 (8, np.float64(0.3779644730092272)),
 (9, np.float64(0.0)),
 (10, np.float64(0.20412414523193154)),
 (11, np.float64(0.5)),
 (12, np.float64(0.0)),
 (13, np.float64(0.17677669529663687)),
 (14, np.float64(0.35355339059327373)),
 (15, np.float64(0.0)),
 (16, np.float64(0.0)),
 (17, np.float64(0.1889822365046136)),
 (18, np.float64(0.22360679774997896)),
 (19, np.float64(0.0)),
 (20, np.float64(0.30151134457776363)),
 (21, np.float64(0.20412414523193154)),
 (22, np.float64(0.1889822365046136)),
 (23, np.float64(0.0)),
 (24, np.float64(0.0)),
 (25, np.float64(0.0)),
 (26, np.float64(0.0)),
 (27, np.float64(0.0)),
 (28, np.float64(0.0)),
 (29, np.float64(0.1889822365046136)),
 (30, np.float64(0.35355339059327373)),
 (31, np.float64(0.0)),
 (32, np.float64(0.0)),

In [17]:
sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
sorted_scores

[(4, np.float64(1.0)),
 (410, np.float64(0.5773502691896258)),
 (43, np.float64(0.5669467095138407)),
 (96, np.float64(0.5303300858899106)),
 (138, np.float64(0.5303300858899106)),
 (195, np.float64(0.5303300858899106)),
 (444, np.float64(0.5303300858899106)),
 (803, np.float64(0.5303300858899106)),
 (11, np.float64(0.5)),
 (59, np.float64(0.5)),
 (68, np.float64(0.5)),
 (71, np.float64(0.5)),
 (97, np.float64(0.5)),
 (330, np.float64(0.5)),
 (378, np.float64(0.5)),
 (514, np.float64(0.5)),
 (647, np.float64(0.5)),
 (738, np.float64(0.5)),
 (947, np.float64(0.5)),
 (991, np.float64(0.5)),
 (811, np.float64(0.45226701686664544)),
 (66, np.float64(0.4472135954999579)),
 (222, np.float64(0.4472135954999579)),
 (234, np.float64(0.4472135954999579)),
 (369, np.float64(0.4472135954999579)),
 (439, np.float64(0.4472135954999579)),
 (463, np.float64(0.4472135954999579)),
 (766, np.float64(0.4472135954999579)),
 (829, np.float64(0.4472135954999579)),
 (399, np.float64(0.4330127018922194)),
 (49

In [18]:
selected_course_index = [i[0] for i in sorted_scores[1:]]
selected_course_index

[410,
 43,
 96,
 138,
 195,
 444,
 803,
 11,
 59,
 68,
 71,
 97,
 330,
 378,
 514,
 647,
 738,
 947,
 991,
 811,
 66,
 222,
 234,
 369,
 439,
 463,
 766,
 829,
 399,
 49,
 54,
 72,
 85,
 107,
 160,
 167,
 205,
 295,
 353,
 402,
 411,
 802,
 953,
 956,
 1002,
 8,
 33,
 35,
 102,
 109,
 113,
 157,
 186,
 363,
 434,
 510,
 650,
 798,
 900,
 14,
 30,
 44,
 46,
 48,
 75,
 89,
 149,
 153,
 200,
 346,
 361,
 366,
 377,
 416,
 451,
 471,
 566,
 628,
 708,
 794,
 864,
 909,
 954,
 963,
 1023,
 1112,
 1115,
 1134,
 1136,
 1141,
 1151,
 62,
 315,
 618,
 683,
 20,
 36,
 201,
 398,
 709,
 99,
 108,
 118,
 302,
 387,
 403,
 468,
 477,
 479,
 538,
 583,
 648,
 652,
 745,
 777,
 823,
 833,
 844,
 863,
 883,
 948,
 962,
 1005,
 1089,
 1171,
 256,
 67,
 77,
 78,
 84,
 88,
 135,
 144,
 164,
 208,
 284,
 299,
 320,
 350,
 356,
 394,
 408,
 429,
 448,
 493,
 500,
 533,
 559,
 570,
 598,
 694,
 696,
 764,
 769,
 815,
 921,
 1012,
 1021,
 1135,
 18,
 50,
 79,
 90,
 141,
 187,
 188,
 196,
 221,
 233,
 248,
 2

In [19]:
selected_course_score = [i[1] for i in sorted_scores[1:]]
selected_course_score 

[np.float64(0.5773502691896258),
 np.float64(0.5669467095138407),
 np.float64(0.5303300858899106),
 np.float64(0.5303300858899106),
 np.float64(0.5303300858899106),
 np.float64(0.5303300858899106),
 np.float64(0.5303300858899106),
 np.float64(0.5),
 np.float64(0.5),
 np.float64(0.5),
 np.float64(0.5),
 np.float64(0.5),
 np.float64(0.5),
 np.float64(0.5),
 np.float64(0.5),
 np.float64(0.5),
 np.float64(0.5),
 np.float64(0.5),
 np.float64(0.5),
 np.float64(0.45226701686664544),
 np.float64(0.4472135954999579),
 np.float64(0.4472135954999579),
 np.float64(0.4472135954999579),
 np.float64(0.4472135954999579),
 np.float64(0.4472135954999579),
 np.float64(0.4472135954999579),
 np.float64(0.4472135954999579),
 np.float64(0.4472135954999579),
 np.float64(0.4330127018922194),
 np.float64(0.4082482904638631),
 np.float64(0.4082482904638631),
 np.float64(0.4082482904638631),
 np.float64(0.4082482904638631),
 np.float64(0.4082482904638631),
 np.float64(0.4082482904638631),
 np.float64(0.4082482904

now, with the help of iloc functions, we will recommend similiar courses

rec_df is new df

In [20]:
rec_df = df.iloc[selected_course_index]
rec_df.head()

,course_id,course_title,url,is_paid,price,num_subscribers,num_reviews,num_lectures,level,content_duration,published_timestamp,subject,profit,published_date,published_time,year,month,day,Clean_title
410,889066,Trading Options Basics,https://www.udemy.com/trading-options-basics/,True,200,8,0,8,Beginner Level,1.5 hours,2016-07-01T03:13:22Z,Business Finance,1600,2016-07-01,03:13:22Z,2016,7,1,Trading Options Basics
43,627540,Options Trading - How to Win with Weekly Options,https://www.udemy.com/work-from-home-setup-you...,True,115,7489,1190,25,Intermediate Level,1 hour,2015-10-22T21:54:28Z,Business Finance,861235,2015-10-22,21:54:28Z,2015,10,22,Options Trading Win Weekly Options
96,474928,Intermediate Options trading concepts for Stoc...,https://www.udemy.com/intermediate-options-tra...,True,40,2000,30,9,All Levels,1 hour,2015-04-13T20:28:04Z,Business Finance,80000,2015-04-13,20:28:04Z,2015,4,13,Intermediate Options trading concepts Stocks O...
138,1243448,Forex Trading with Fixed 'Risk through Options...,https://www.udemy.com/forexoptions/,True,200,611,4,26,Beginner Level,1 hour,2017-06-07T17:15:07Z,Business Finance,122200,2017-06-07,17:15:07Z,2017,6,7,Forex Trading Fixed Risk Options Trading
195,919906,Trading Options For Consistent Returns: Option...,https://www.udemy.com/trading-options-for-income/,False,0,4077,281,20,Beginner Level,1.5 hours,2016-08-18T21:57:04Z,Business Finance,0,2016-08-18,21:57:04Z,2016,8,18,Trading Options Consistent Returns Options Basics


now, we will create another column Similarity score(that will contain selected course score

In [21]:
rec_df['Similarity_score'] = selected_course_score

In [22]:
final_recommended_courses = rec_df[['course_title',\
                        'Similarity_score', 'url', 'price',\
                        'num_subscribers']]
final_recommended_courses

,course_title,Similarity_score,url,price,num_subscribers
410,Trading Options Basics,0.577350,https://www.udemy.com/trading-options-basics/,200,8
43,Options Trading - How to Win with Weekly Options,0.566947,https://www.udemy.com/work-from-home-setup-you...,115,7489
96,Intermediate Options trading concepts for Stoc...,0.530330,https://www.udemy.com/intermediate-options-tra...,40,2000
138,Forex Trading with Fixed 'Risk through Options...,0.530330,https://www.udemy.com/forexoptions/,200,611
195,Trading Options For Consistent Returns: Option...,0.530330,https://www.udemy.com/trading-options-for-income/,0,4077
...,...,...,...,...,...
3678,Learn jQuery from Scratch - Master of JavaScri...,0.000000,https://www.udemy.com/easy-jquery-for-beginner...,100,1040
3679,How To Design A WordPress Website With No Codi...,0.000000,https://www.udemy.com/how-to-make-a-wordpress-...,25,306
3680,Learn and Build using Polymer,0.000000,https://www.udemy.com/learn-and-build-using-po...,40,513
3681,CSS Animations: Create Amazing Effects on Your...,0.000000,https://www.udemy.com/css-animations-create-am...,50,300


In [23]:
final_recommended_courses = final_recommended_courses[final_recommended_courses['Similarity_score']>0.5]
final_recommended_courses

,course_title,Similarity_score,url,price,num_subscribers
410,Trading Options Basics,0.577350,https://www.udemy.com/trading-options-basics/,200,8
43,Options Trading - How to Win with Weekly Options,0.566947,https://www.udemy.com/work-from-home-setup-you...,115,7489
96,Intermediate Options trading concepts for Stoc...,0.530330,https://www.udemy.com/intermediate-options-tra...,40,2000
138,Forex Trading with Fixed 'Risk through Options...,0.530330,https://www.udemy.com/forexoptions/,200,611
195,Trading Options For Consistent Returns: Option...,0.530330,https://www.udemy.com/trading-options-for-income/,0,4077
444,The Advantages of ETF Options and Index Option...,0.530330,https://www.udemy.com/learn-etf-options-and-in...,60,52
803,Options Spreads Bundle- the heart of Options ...,0.530330,https://www.udemy.com/options-spreads-explained/,120,623
